[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/22_conv2d.ipynb)

# 🟠 Medium: 2D Convolution

Implement **2D convolution** from scratch.

### Signature
```python
def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    # x: (B, C_in, H, W), weight: (C_out, C_in, kH, kW)
    # Returns: (B, C_out, H_out, W_out)
```

### Rules
- Do NOT use `F.conv2d` or `nn.Conv2d`
- Support `stride` and `padding` parameters
- `F.pad` for zero-padding is allowed

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.8 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn.functional as F

In [5]:
# ✏️ YOUR IMPLEMENTATION HERE

def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    # 1. 获取输入的维度形状
    # x: (B, C_in, H, W)
    # weight: (C_out, C_in, kH, kW)
    B, C_in, H, W = x.shape
    C_out, _, kH, kW = weight.shape

    # 2. 处理步长 (stride) 和 填充 (padding) 的单值或元组情况
    sh, sw = (stride, stride) if isinstance(stride, int) else stride
    ph, pw = (padding, padding) if isinstance(padding, int) else padding

    # 3. 使用 F.pad 进行零填充
    # F.pad 的默认填充顺序是：左、右、上、下
    x_padded = F.pad(x, (pw, pw, ph, ph), mode='constant', value=0)

    # 4. 计算输出的 H 和 W
    H_out = (H + 2 * ph - kH) // sh + 1
    W_out = (W + 2 * pw - kW) // sw + 1

    # 5. 初始化输出张量
    out = torch.zeros((B, C_out, H_out, W_out), dtype=x.dtype, device=x.device)

    # 6. 滑动窗口计算卷积
    for b in range(B): # 遍历 Batch
        for c_o in range(C_out): # 遍历输出通道
            for h_o in range(H_out): # 遍历输出的高度坐标
                for w_o in range(W_out): # 遍历输出的宽度坐标

                    # 计算当前窗口在 x_padded 上的起始和结束位置
                    h_start = h_o * sh
                    h_end = h_start + kH
                    w_start = w_o * sw
                    w_end = w_start + kW

                    # 提取当前滑动窗口对应的局部区域形状为 (C_in, kH, kW)
                    x_window = x_padded[b, :, h_start:h_end, w_start:w_end]

                    # 对应通道和位置相乘并求和 (点积)
                    # weight[c_o] 的形状也是 (C_in, kH, kW)
                    out[b, c_o, h_o, w_o] = torch.sum(x_window * weight[c_o])

            # 7. 如果有偏置，加上偏置
            if bias is not None:
                out[b, c_o, :, :] += bias[c_o]

    return out

In [6]:
# 🧪 Debug
x = torch.randn(1, 3, 8, 8)
w = torch.randn(16, 3, 3, 3)
print('Output:', my_conv2d(x, w).shape)
print('Match:', torch.allclose(my_conv2d(x, w), F.conv2d(x, w), atol=1e-4))

Output: torch.Size([1, 16, 6, 6])
Match: True


In [8]:
# ✅ SUBMIT
from torch_judge import check, hint
check('conv2d')
hint("conv2d")


🧪 Testing: 2D Convolution (Medium)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (93.9ms)
  ✅ [2/5] Matches F.conv2d (167.0ms)
  ✅ [3/5] With padding (8.7ms)
  ✅ [4/5] With stride (1.9ms)
  ✅ [5/5] Gradient flow (19.8ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (291.2ms total)
  Progress saved. Run status() to see your dashboard.


💡 Hint for 2D Convolution:
   Extract patches using unfold or nested loops. For each output position, sum(patch * kernel). Support stride and padding (zero-pad with F.pad).

